# MOUSE CORE · BGE-M3 + strukturální pilot · opravený vstup Kocoura

Pilot tří původních karet: **Myš, Ryba, Kocour**. Texty zůstávají celé a beze změn.

Notebook počítá dvě odlišné vrstvy:
1. **Pozorovatelné vlastnosti textu**: opakování slov a dvojic, lexikální rozmanitost, četnosti interpunkce, pozice a kontexty opakovaných prvků.
2. **Vektorovou geometrii**: jeden skutečný dense embedding BGE-M3 na celou pohádku a párové vzdálenosti mezi třemi vektory.

Důležité metodické hranice:
- Žádná pohádka se nedělí na významové segmenty.
- Kontexty výskytů jsou pouze výřezy pro kontrolu, nikoli nové embeddingové jednotky.
- BGE-M3 zpracovává pouze text. Ilustrace jsou připojené jako metadata, nikoli obrazové vektory.
- Tři body dovolují spočítat vzájemné vzdálenosti a zobrazit trojúhelník, ale **nestačí k odvození topologie obecného významového prostoru**.
- Četnost nebo podobnost sama o sobě nedokazuje významovou funkci. Tu je nutné interpretovat s dokladem v kontextu.

V Kaggle zapni Internet a GPU a spusť **Run All**. Model se stáhne do dočasného prostředí Kaggle, nikoli do telefonu.

In [ ]:
!pip -q install -U FlagEmbedding pandas matplotlib

In [ ]:
import json, re, math, unicodedata
from pathlib import Path
from urllib.request import urlopen
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

REPO = "https://raw.githubusercontent.com/artmisanovak-prog/generative_gen/main"
MODEL_NAME = "BAAI/bge-m3"
ANGLE_LENSES = ["Analytický", "Kulturní", "Filozofický", "Naivní", "Jazykový", "Metaforický", "Asociační", "Introspektivní", "Ironický", "Groteskní", "Absurdní", "Surrealistický"]
CARDS = {
    "mys": {"name": "Myš", "text": "doma/primarni_data/kriva_pohadka/mys.txt", "image": "doma/primarni_data/ilustrace/mys.jpg"},
    "ryba": {"name": "Ryba", "text": "doma/primarni_data/kriva_pohadka/ryba.txt", "image": "doma/primarni_data/ilustrace/ryba.jpg"},
    "kocour": {"name": "Kocour", "text": "doma/primarni_data/kriva_pohadka/kocour.txt", "image": "doma/primarni_data/ilustrace/kocour.jpg"}
}
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
print("Zařízení:", "GPU" if torch.cuda.is_available() else "CPU")
print("Výstup:", OUT)

## 1 · Načtení původních dat

Stahujeme přesné zdrojové soubory z repozitáře. Nepřepisujeme je, nepřekládáme a nekrátíme.

In [ ]:
documents = []
input_normalization = {}
for card_id, info in CARDS.items():
    raw = urlopen(f"{REPO}/{info['text']}").read().decode("utf-8-sig")
    model_text = raw.strip()
    if card_id == "kocour":
        # Odstraníme pouze technické metadata řádky na začátku souboru.
        # Vlastní nadpis, podnadpis i celý příběh zůstávají beze změny.
        cleaned = re.sub(
            r"\AKarta:[^\n]*\n\s*Text:[^\n]*\n+",
            "",
            model_text,
            count=1
        )
        assert cleaned != model_text, "Očekávané technické záhlaví Kocoura nebylo nalezeno."
        assert cleaned.lstrip().startswith("O KOCOUROVI"), "Po očištění nezačíná Kocour očekávaným nadpisem."
        model_text = cleaned.strip()
        input_normalization[card_id] = {
            "removed": ["Karta: ...", "Text: ..."],
            "reason": "Technické metadata, nikoli součást pohádky",
            "story_heading_preserved": True,
            "story_body_preserved": True
        }
    else:
        input_normalization[card_id] = {
            "removed": [],
            "reason": "Bez automatického odstraňování; celý zdrojový text zachován"
        }
    documents.append({
        "card_id": card_id,
        "card_name": info["name"],
        "source_file": info["text"],
        "illustration_file": info["image"],
        "illustration_url": f"{REPO}/{info['image']}",
        "original_text": model_text
    })
    print(f"{info['name']}: {len(model_text)} znaků po přípravě | {info['text']}")
assert len(documents) == 3
assert len({d['card_id'] for d in documents}) == 3
print("Úpravy vstupů:", input_normalization)


## 2 · Strukturální měření bez segmentace

Tato část neříká, co slovo znamená. Měří, co lze ověřit přímo v textu: kolikrát se jazykový prvek vrací, jaká je pestrost slovníku, jaké znaky se opakují a v jakém bezprostředním okolí se výskyty nacházejí.

Kontexty jsou pouze okolní text kolem konkrétního výskytu. Nejsou to automaticky významové jednotky.

In [ ]:
WORD_RE = re.compile(r"[^\W\d_]+(?:[’'-][^\W\d_]+)*", re.UNICODE)
BIGRAM_RE = None

def contexts_for(text, spans, radius=75):
    out = []
    for start, end, token in spans:
        out.append({
            "token": token,
            "start_char": start,
            "end_char": end,
            "context": text[max(0, start-radius):min(len(text), end+radius)]
        })
    return out

def analyze_structure(doc):
    text = doc["original_text"]
    matches = list(WORD_RE.finditer(text))
    words = [m.group(0).casefold() for m in matches]
    counts = Counter(words)
    bigrams = Counter(zip(words, words[1:]))
    punct = Counter(ch for ch in text if unicodedata.category(ch).startswith("P"))
    total = len(words)
    unique = len(counts)
    probs = [v/total for v in counts.values()] if total else []
    entropy = -sum(p*math.log2(p) for p in probs) if probs else 0.0
    max_entropy = math.log2(unique) if unique > 1 else 0.0
    repeated = {w:c for w,c in counts.items() if c > 1}
    repeated_spans = [(m.start(), m.end(), m.group(0)) for m in matches if counts[m.group(0).casefold()] > 1]
    repeated_bigrams = [{"bigram": list(k), "count": v} for k,v in bigrams.items() if v > 1]
    punctuation_contexts = []
    for i,ch in enumerate(text):
        if ch in ".!?…,:;":
            punctuation_contexts.append({
                "mark": ch, "start_char": i,
                "context": text[max(0,i-75):min(len(text),i+76)]
            })
    sentence_like = len(re.findall(r"[.!?…]+", text))
    paragraphs = [p for p in re.split(r"\n\s*\n", text) if p.strip()]
    return {
        "card_id": doc["card_id"],
        "card_name": doc["card_name"],
        "source_file": doc["source_file"],
        "characters_including_spaces": len(text),
        "characters_without_whitespace": sum(not c.isspace() for c in text),
        "word_tokens": total,
        "unique_word_forms_casefolded": unique,
        "type_token_ratio": round(unique/total, 6) if total else 0,
        "hapax_count": sum(1 for c in counts.values() if c == 1),
        "hapax_ratio": round(sum(1 for c in counts.values() if c == 1)/unique, 6) if unique else 0,
        "repeated_token_types": len(repeated),
        "token_share_from_repeated_types": round(sum(c for c in counts.values() if c > 1)/total, 6) if total else 0,
        "word_frequency": counts.most_common(30),
        "repeated_word_occurrences_with_context": contexts_for(text, repeated_spans),
        "repeated_adjacent_bigrams": repeated_bigrams,
        "punctuation_counts": dict(punct),
        "selected_punctuation_contexts": punctuation_contexts,
        "sentence_ending_mark_count": sentence_like,
        "paragraph_count": len(paragraphs),
        "word_frequency_entropy_bits": round(entropy, 6),
        "normalized_word_frequency_entropy": round(entropy/max_entropy, 6) if max_entropy else 0
    }

structure = [analyze_structure(d) for d in documents]
summary_rows = [{
    "card": x["card_name"],
    "characters": x["characters_including_spaces"],
    "word_tokens": x["word_tokens"],
    "unique_forms": x["unique_word_forms_casefolded"],
    "type_token_ratio": x["type_token_ratio"],
    "repeated_types": x["repeated_token_types"],
    "repeated_token_share": x["token_share_from_repeated_types"],
    "paragraphs": x["paragraph_count"],
    "sentence_end_marks": x["sentence_ending_mark_count"]
} for x in structure]
display(pd.DataFrame(summary_rows).set_index("card"))


## 3 · Skutečné embeddingy celých pohádek

Každý text je předán modelu jako jedna položka. Model může interně tokenizovat text, ale my z toho neděláme vlastní sémantické segmenty.

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(MODEL_NAME, use_fp16=torch.cuda.is_available())
whole_texts = [d["original_text"] for d in documents]
encoded = model.encode(
    whole_texts,
    batch_size=3,
    max_length=8192,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False
)
vectors = np.asarray(encoded["dense_vecs"], dtype=np.float32)
print("Počet vektorů:", len(vectors))
print("Dimenze:", vectors.shape[1])
assert vectors.shape[0] == 3

## 4 · Geometrie tří reprezentací

Počítáme párové vztahy v prostoru embeddingů. Kosinová podobnost měří orientační podobnost směrů vektorů. Úhlová vzdálenost převádí kosinus na úhel. Eukleidovská vzdálenost měří přímou vzdálenost vektorů.

Žádná z těchto hodnot sama o sobě není měřítkem lidského porozumění. Výsledky platí pro tento model, jeho verzi a tento způsob reprezentace.

In [ ]:
def cosine(a, b):
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a,b)/den) if den else float("nan")

geometry = []
cos_matrix = np.zeros((3,3), dtype=float)
euclidean_matrix = np.zeros((3,3), dtype=float)
angular_matrix = np.zeros((3,3), dtype=float)

for i in range(3):
    for j in range(3):
        c = cosine(vectors[i], vectors[j])
        cos_matrix[i,j] = c
        euclidean_matrix[i,j] = float(np.linalg.norm(vectors[i]-vectors[j]))
        angular_matrix[i,j] = float(np.arccos(np.clip(c,-1,1)))

for i in range(3):
    for j in range(i+1,3):
        geometry.append({
            "card_a": documents[i]["card_id"],
            "card_b": documents[j]["card_id"],
            "cosine_similarity": float(cos_matrix[i,j]),
            "cosine_distance": float(1-cos_matrix[i,j]),
            "angular_distance_radians": float(angular_matrix[i,j]),
            "angular_distance_degrees": float(np.degrees(angular_matrix[i,j])),
            "euclidean_distance": float(euclidean_matrix[i,j])
        })

labels = [d["card_name"] for d in documents]
display(pd.DataFrame(cos_matrix, index=labels, columns=labels).round(5))
display(pd.DataFrame(geometry))


In [ ]:
fig, ax = plt.subplots(figsize=(6,5))
im = ax.imshow(cos_matrix, vmin=-1, vmax=1)
ax.set_xticks(range(3), labels)
ax.set_yticks(range(3), labels)
ax.set_title("BGE-M3 · kosinová podobnost celých pohádek")
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cos_matrix[i,j]:.3f}", ha="center", va="center")
fig.colorbar(im, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()

# Zobrazení tří bodů jako trojúhelníku podle jejich eukleidovských vzdáleností.
# Souřadnice jsou pouze geometrická pomůcka; osy nemají samostatný sémantický význam.
d01 = euclidean_matrix[0,1]
d02 = euclidean_matrix[0,2]
d12 = euclidean_matrix[1,2]
if d01 > 1e-12:
    x2 = (d02**2 + d01**2 - d12**2)/(2*d01)
    y2 = math.sqrt(max(0.0, d02**2 - x2**2))
    coords = [(0,0),(d01,0),(x2,y2)]
    fig, ax = plt.subplots(figsize=(6,5))
    for (x,y), label in zip(coords, labels):
        ax.scatter(x,y)
        ax.annotate(label, (x,y), xytext=(6,6), textcoords="offset points")
    for i,j in [(0,1),(0,2),(1,2)]:
        ax.plot([coords[i][0],coords[j][0]],[coords[i][1],coords[j][1]], alpha=.65)
    ax.set_title("Trojúhelník vzdáleností ve vektorovém prostoru")
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel("Pomocná geometrická osa")
    ax.set_ylabel("Pomocná geometrická osa")
    plt.tight_layout()
    plt.show()

## 5 · Export do MOUSE CORE

Vzniknou tři soubory:
- `mouse_core_bge_m3_vectors.json` – tři skutečné embeddingy pro Vector Lab.
- `mouse_core_bge_m3_analysis.json` – strukturální měření, kontexty opakování, párová geometrie a metadata.
- `mouse_core_bge_m3_manifest.json` – popis běhu, modelu a zdrojů.

Původní texty jsou přiložené jako zdrojové podklady. Žádné automatické významové štítky se nevydávají za fakta.

In [ ]:
vector_rows = []
for doc, vec in zip(documents, vectors):
    vector_rows.append({
        "id": doc["card_id"] + "-full",
        "card_id": doc["card_id"],
        "card_name": doc["card_name"],
        "kind": "full_story",
        "text": doc["original_text"],
        "source_file": doc["source_file"],
        "illustration_file": doc["illustration_file"],
        "illustration_url": doc["illustration_url"],
        "model": MODEL_NAME,
        "vector": [float(v) for v in vec]
    })

manifest = {
    "experiment": "MOUSE CORE BGE-M3 + structural pilot",
    "model": MODEL_NAME,
    "model_type": "text_embedding",
    "language": "Czech",
    "cards": [d["card_id"] for d in documents],
    "angle_lenses_defined_by_method": ANGLE_LENSES,
    "angle_lenses_scored_automatically": False,
    "item_count": len(vector_rows),
    "vector_dimension": int(vectors.shape[1]),
    "source_repository": "https://github.com/artmisanovak-prog/generative_gen",
    "segmentation": "none; each complete original story is one embedding input",
    "input_normalization": input_normalization,
    "geometry_scope": "pairwise geometry of three points only; not a general semantic topology",
    "illustrations": "metadata references only; no image embeddings",
    "limitations": [
        "Similarity is not proof of semantic understanding.",
        "Lexical counts do not determine the function or meaning of a repeated element.",
        "Three stories are a pilot, not a statistically representative sample."
    ]
}

analysis_payload = {
    "schema": "mouse-core-bge-m3-analysis-v1",
    "manifest": manifest,
    "structural_analysis": structure,
    "embedding_geometry": {
        "model": MODEL_NAME,
        "labels": labels,
        "cosine_similarity_matrix": cos_matrix.tolist(),
        "euclidean_distance_matrix": euclidean_matrix.tolist(),
        "angular_distance_matrix_radians": angular_matrix.tolist(),
        "pairwise": geometry,
        "interpretation_note": "Only pairwise relations among these three complete-story vectors are measured. No general topology is inferred."
    }
}

vectors_path = OUT / "mouse_core_bge_m3_vectors.json"
analysis_path = OUT / "mouse_core_bge_m3_analysis.json"
manifest_path = OUT / "mouse_core_bge_m3_manifest.json"

vectors_path.write_text(json.dumps(vector_rows, ensure_ascii=False), encoding="utf-8")
analysis_path.write_text(json.dumps(analysis_payload, ensure_ascii=False, indent=2), encoding="utf-8")
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("Uloženo:")
print(vectors_path)
print(analysis_path)
print(manifest_path)


## 6 · Kontrola

Notebook ověří tři celé pohádky, shodu dimenzí a úplnost párových vzdáleností. Pokud kontrola projde, stáhni oba JSON soubory a načti je do MOUSE CORE.

In [ ]:
assert len(vector_rows) == 3
assert all(len(x["vector"]) == vectors.shape[1] for x in vector_rows)
assert len(geometry) == 3
assert [x["card_id"] for x in vector_rows] == ["mys", "ryba", "kocour"]
assert all(x["kind"] == "full_story" for x in vector_rows)
assert np.allclose(cos_matrix, cos_matrix.T)
assert np.allclose(euclidean_matrix, euclidean_matrix.T)
print("Kontrola prošla.")
print("3 celé pohádky · 3 skutečné vektory · 3 párové vztahy.")
print("Výstupy: mouse_core_bge_m3_vectors.json, mouse_core_bge_m3_analysis.json, mouse_core_bge_m3_manifest.json")